In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

PROC_DIR = Path("../data/processed")

expr = pd.read_csv(PROC_DIR / "expression_filtered.csv", index_col=0)
drug = pd.read_csv(PROC_DIR / "drug_final.csv")

print("expression:", expr.shape)
print("drug:", drug.shape)
print("unique drugs:", drug['inhibitor'].nunique())

expression: (318, 14121)
drug: (34764, 5)
unique drugs: 122


In [2]:
# define the per-drug modelling function
from sklearn.linear_model import ElasticNet

def train_drug_model(drug_name, drug_df, expr_df, test_size=0.2, random_state=42):
    subset = drug_df[drug_df['inhibitor'] == drug_name][['cbio_patient_id', 'auc']].drop_duplicates()
    subset = subset[subset['cbio_patient_id'].isin(expr_df.index)]
    
    X = expr_df.loc[subset['cbio_patient_id']].values
    y = subset['auc'].values
    
    if len(y) < 30:
        return None
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
    en.fit(X_train, y_train)
    y_pred_en = en.predict(X_test)
    r_en, _ = pearsonr(y_test, y_pred_en)
    rmse_en = np.sqrt(mean_squared_error(y_test, y_pred_en))
    
    xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                   subsample=0.8, colsample_bytree=0.3,
                                   random_state=random_state, n_jobs=-1, device='cpu')
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    r_xgb, _ = pearsonr(y_test, y_pred_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    
    return {
        'drug': drug_name,
        'n_patients': len(y),
        'n_test': len(y_test),
        'pearson_en': round(r_en, 4),
        'rmse_en': round(rmse_en, 4),
        'pearson_xgb': round(r_xgb, 4),
        'rmse_xgb': round(rmse_xgb, 4)
    }

print("Function redefined.")

print("done")

Function redefined.
done


In [3]:
# run across all 122 drugs

results = []
drugs = drug['inhibitor'].unique()

for i, drug_name in enumerate(drugs):
    result = train_drug_model(drug_name, drug, expr)
    if result is not None:
        results.append(result)
    if (i + 1) % 20 == 0:
        print(f"processing: {i+1}/{len(drugs)} drugs done")

results_df = pd.DataFrame(results)
results_df.to_csv(Path("../results/baseline_results.csv"), index=False)

print(f"\ncompleted: {len(results_df)} / {len(drugs)} drugs")
print("\nElastic Net:")
print(f"  median pearson r: {results_df['pearson_en'].median():.4f}")
print(f"  median RMSE:      {results_df['rmse_en'].median():.4f}")
print("\nXGBoost:")
print(f"  median pearson r: {results_df['pearson_xgb'].median():.4f}")
print(f"  median RMSE:      {results_df['rmse_xgb'].median():.4f}")

processing: 20/122 drugs done
processing: 40/122 drugs done
processing: 60/122 drugs done
processing: 80/122 drugs done
processing: 100/122 drugs done
processing: 120/122 drugs done

completed: 122 / 122 drugs

Elastic Net:
  median pearson r: 0.2915
  median RMSE:      46.7373

XGBoost:
  median pearson r: 0.2813
  median RMSE:      43.3680


In [4]:
# deeper look at distribution results

print("Elastic Net Pearson R distribution:")
print(results_df['pearson_en'].describe())
print(f"  Drugs with R > 0.4: {(results_df['pearson_en'] > 0.4).sum()}")
print(f"  Drugs with R > 0.3: {(results_df['pearson_en'] > 0.3).sum()}")
print(f"  Drugs with R < 0.0: {(results_df['pearson_en'] < 0.0).sum()}")

print("\nXGBoost Pearson R distribution:")
print(results_df['pearson_xgb'].describe())
print(f"  Drugs with R > 0.4: {(results_df['pearson_xgb'] > 0.4).sum()}")
print(f"  Drugs with R > 0.3: {(results_df['pearson_xgb'] > 0.3).sum()}")
print(f"  Drugs with R < 0.0: {(results_df['pearson_xgb'] < 0.0).sum()}")

print("\nTop 5 drugs by XGBoost Pearson R:")
print(results_df.nlargest(5, 'pearson_xgb')[['drug', 'pearson_en', 'pearson_xgb', 'n_patients']])

Elastic Net Pearson R distribution:
count    122.000000
mean       0.277130
std        0.152077
min       -0.055700
25%        0.171475
50%        0.291500
75%        0.390650
max        0.587900
Name: pearson_en, dtype: float64
  Drugs with R > 0.4: 28
  Drugs with R > 0.3: 57
  Drugs with R < 0.0: 5

XGBoost Pearson R distribution:
count    122.000000
mean       0.282313
std        0.170825
min       -0.234200
25%        0.184250
50%        0.281300
75%        0.400475
max        0.635400
Name: pearson_xgb, dtype: float64
  Drugs with R > 0.4: 31
  Drugs with R > 0.3: 57
  Drugs with R < 0.0: 7

Top 5 drugs by XGBoost Pearson R:
                        drug  pearson_en  pearson_xgb  n_patients
82              Panobinostat      0.3788       0.6354         151
102                Sorafenib      0.4734       0.6229         352
111  Trametinib (GSK1120212)      0.5729       0.6072         334
0      17-AAG (Tanespimycin)      0.5097       0.5883         314
104                Sunitinib   

In [6]:
# save models for conformal prediction & plot results

from sklearn.linear_model import ElasticNet
import pickle

Path("../results").mkdir(exist_ok=True)
Path("../results/models").mkdir(exist_ok=True)

drugs_list = drug['inhibitor'].unique()
models = {}

for drug_name in drugs_list:
    subset = drug[drug['inhibitor'] == drug_name][['cbio_patient_id', 'auc']].drop_duplicates()
    subset = subset[subset['cbio_patient_id'].isin(expr.index)]
    X = expr.loc[subset['cbio_patient_id']].values
    y = subset['auc'].values
    if len(y) < 30:
        continue
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
    en.fit(X_train, y_train)
    
    xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                   subsample=0.8, colsample_bytree=0.3,
                                   random_state=42, n_jobs=-1, device='cpu')
    xgb_model.fit(X_train, y_train)
    
    models[drug_name] = {'en': en, 'xgb': xgb_model,
                          'X_test': X_test, 'y_test': y_test,
                          'X_train': X_train, 'y_train': y_train}

with open("../results/models/trained_models.pkl", "wb") as f:
    pickle.dump(models, f)

print(f"Saved {len(models)} drug models.")

Saved 122 drug models.


In [7]:
import pickle
from pathlib import Path

model_path = Path("../results/models/trained_models.pkl")
print("File exists:", model_path.exists())
if model_path.exists():
    print("File size:", model_path.stat().st_size / 1e6, "MB")
    with open(model_path, "rb") as f:
        models = pickle.load(f)
    print("Models saved:", len(models))
    print("Sample drugs:", list(models.keys())[:5])

File exists: True
File size: 3953.524955 MB
Models saved: 122
Sample drugs: ['17-AAG (Tanespimycin)', 'A-674563', 'ABT-737', 'AT7519', 'AZD1480']
